# External Data Collection: Exchange Rates

This notebook retrieves historical exchange rate data used as additional
explanatory variables in the MAANG stock analysis.

Exchange rates are incorporated as macroeconomic indicators that may
influence stock price dynamics. The retrieved data will later be merged
with the primary financial dataset prepared in the previous notebook.

## Imports

Libraries used for retrieving and processing exchange rate data.

In [1]:
from bs4 import BeautifulSoup as BS
import requests
import pandas as pd
import re

## Retrieve Exchange Rate Data

Send a request to the external API and retrieve exchange rate time series.

In [2]:
df = pd.DataFrame()

URLS_1 = ['https://www.exchange-rates.org/converter/usd-eur', 'https://www.exchange-rates.org/converter/eur-usd', 
         'https://www.exchange-rates.org/converter/uah-usd', 'https://www.exchange-rates.org/converter/usd-uah', 
         'https://www.exchange-rates.org/converter/eur-uah', 'https://www.exchange-rates.org/converter/uah-eur']

for URL_1 in URLS_1:
    
    from_cur = URL_1[-7:].split('-')[0].upper()
    to_cur = URL_1[-7:].split('-')[1].upper()
    
    req_1 = requests.get(URL_1)
    soup = BS(req_1.text, "html.parser")

    orig_path = "https://www.exchange-rates.org"

    urls = [orig_path + i[:-6] for i in re.findall(r'/\S*\d', str(soup.find_all('ul', class_='rates-by-year')[0]))]

    df_for_one = pd.DataFrame()

    for url in urls:
        req_2 = requests.get(url)
        soup_2 = BS(req_2.text, "html.parser")
        dates = [pd.to_datetime(date.text) for date in soup_2.find_all('a', class_='n')]
        values = [float(val.text.split()[-2]) for val in soup_2.find_all('span', class_='n')]
        _ = pd.DataFrame(values, index=dates, columns=[f'1_{from_cur}_to_{to_cur}'])
        df_for_one = pd.concat([df_for_one,_], axis=0)
    
    df_for_one = df_for_one.sort_index()
    display(df_for_one.tail())
        
    df = pd.concat([df, df_for_one], axis=1)

,1_USD_to_EUR
2024-06-10,0.9289
2024-06-11,0.9309
2024-06-12,0.9249
2024-06-13,0.9312
2024-06-14,0.9326


,1_EUR_to_USD
2024-06-10,1.0766
2024-06-11,1.0743
2024-06-12,1.0812
2024-06-13,1.0739
2024-06-14,1.0723


,1_UAH_to_USD
2024-06-10,0.02473
2024-06-11,0.02468
2024-06-12,0.02470
2024-06-13,0.02449
2024-06-14,0.02458


,1_USD_to_UAH
2024-06-10,40.439
2024-06-11,40.512
2024-06-12,40.483
2024-06-13,40.835
2024-06-14,40.682


,1_EUR_to_UAH
2024-06-10,43.536
2024-06-11,43.520
2024-06-12,43.770
2024-06-13,43.854
2024-06-14,43.621


,1_UAH_to_EUR
2024-06-10,0.02297
2024-06-11,0.02298
2024-06-12,0.02285
2024-06-13,0.02280
2024-06-14,0.02292


## Convert Data to DataFrame

Transform the raw API response into a structured pandas DataFrame
suitable for analysis and merging with the stock dataset.

In [3]:
df = df.dropna().sort_index()
df.index.name = 'date'
df

,1_USD_to_EUR,1_EUR_to_USD,1_UAH_to_USD,1_USD_to_UAH,1_EUR_to_UAH,1_UAH_to_EUR
date,,,,,,
2015-01-01,0.8265,1.2098,0.06322,15.819,19.138,0.05225
2015-01-02,0.8324,1.2014,0.06322,15.819,19.004,0.05262
2015-01-03,0.8335,1.1997,0.06322,15.819,18.978,0.05269
2015-01-04,0.8370,1.1946,0.06265,15.962,18.900,0.05243
2015-01-05,0.8391,1.1918,0.06322,15.818,18.852,0.05305
...,...,...,...,...,...,...
2024-06-10,0.9289,1.0766,0.02473,40.439,43.536,0.02297
2024-06-11,0.9309,1.0743,0.02468,40.512,43.520,0.02298
2024-06-12,0.9249,1.0812,0.02470,40.483,43.770,0.02285


In [4]:
df.to_csv('/diploma_info/datalake/currency_rates.csv')

## Output

The resulting dataset contains daily exchange rate values that can be
merged with the main MAANG stock dataset.

These external variables are used as additional features in subsequent
modelling notebooks.